In [1]:
# Step 1: Import Libraries
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import joblib

# Step 2: Load Dataset
df = pd.read_csv(r"C:\Users\NIYATI RAJUKUMAR\OneDrive\Desktop\fake.csv\fake.csv") 
print("Original Data Sample:")
print(df[['text', 'type']].head())

# Step 3: Convert multi-class 'type' into binary 'label'
fake_types = ['fake', 'bs', 'conspiracy', 'junksci', 'satire', 'hate']
df['label'] = df['type'].apply(lambda x: 'fake' if x in fake_types else 'real')

print("\nLabel Distribution:")
print(df['label'].value_counts())

# Step 4: Text Preprocessing
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub('[^a-zA-Z\s]', '', text)  # Remove punctuation/numbers
    return text

df['text'] = df['text'].apply(preprocess_text)

# Step 5: Split into Train/Test Sets
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 6: TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Step 7: Train Naive Bayes Model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# Step 8: Predict and Evaluate
y_pred = model.predict(X_test_tfidf)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=1))

# Step 9: Save the Model and Vectorizer
joblib.dump(model, 'news_detection_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("\n✅ Model and Vectorizer saved as 'news_detection_model.pkl' and 'tfidf_vectorizer.pkl'")

# Step 10: Test the Saved Model
# Load the model and vectorizer
loaded_model = joblib.load('news_detection_model.pkl')
loaded_vectorizer = joblib.load('tfidf_vectorizer.pkl')

# Test with a new sample text
sample_text = ["Breaking news: Scientists find cure for cancer!"]
sample_tfidf = loaded_vectorizer.transform(sample_text)
prediction = loaded_model.predict(sample_tfidf)

print(f"\n🔎 Sample Prediction: {'FAKE' if prediction[0] == 'fake' else 'REAL'}")


Original Data Sample:
                                                text  type
0  Print They should pay all the back all the mon...  bias
1  Why Did Attorney General Loretta Lynch Plead T...  bias
2  Red State : \nFox News Sunday reported this mo...  bias
3  Email Kayla Mueller was a prisoner and torture...  bias
4  Email HEALTHCARE REFORM TO MAKE AMERICA GREAT ...  bias

Label Distribution:
label
fake    12435
real      564
Name: count, dtype: int64

Classification Report:
              precision    recall  f1-score   support

        fake       0.96      1.00      0.98      2487
        real       1.00      0.00      0.00       113

    accuracy                           0.96      2600
   macro avg       0.98      0.50      0.49      2600
weighted avg       0.96      0.96      0.94      2600


✅ Model and Vectorizer saved as 'news_detection_model.pkl' and 'tfidf_vectorizer.pkl'

🔎 Sample Prediction: FAKE
